In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import networkx as nx
from jax.experimental.sparse import BCOO

In [64]:
import torch
import numpy as np

def lsh_partition(vectors, num_hashes=10, hash_size=32, seed=42):
    """
    Partition vectors into buckets using Locality Sensitive Hashing (LSH).
    Args:
        vectors (torch.Tensor): Input vectors (n_vectors x dim).
        num_hashes (int): Number of hash functions.
        hash_size (int): Length of each hash vector.
        seed (int): Random seed for reproducibility.
    Returns:
        buckets (dict): A dictionary where keys are hash values, and values are lists of vector indices.
    """
    torch.manual_seed(seed)
    random_planes = torch.randn((num_hashes, vectors.size(1), hash_size), device=vectors.device)
    projections = (vectors @ random_planes) # Binary hashing
    hash_keys = projections.sum(dim=2).sign().transpose(0,1)
    hash_keys = hash_keys.cpu().numpy().astype(np.int32)

    # Group vectors into buckets
    buckets = {}
    for idx, key in enumerate(hash_keys):
        key_tuple = tuple(key)
        if key_tuple not in buckets:
            buckets[key_tuple] = []
        buckets[key_tuple].append(idx)
    return buckets, hash_keys

def compute_sparse_distances_from_buckets(vectors, buckets, threshold_quantile=0.98):
    """
    Compute sparse distance matrix by processing buckets.
    Args:
        vectors (torch.Tensor): Input vectors (n_vectors x dim).
        buckets (dict): LSH buckets.
        threshold (float): Distance threshold for sparsity.
    Returns:
        sparse_matrix (torch.sparse.Tensor): Sparse distance matrix.
    """
    indices = []
    values = []

    for bucket_indices in buckets.values():
        if len(bucket_indices) < 2:
            continue  # Skip buckets with fewer than 2 vectors

        # Extract vectors in this bucket
        bucket_vectors = vectors[bucket_indices]
        
        # Compute pairwise distances within the bucket
        distances = torch.cdist(bucket_vectors, bucket_vectors, p=2)
        # print(distances.shape)
        # print(torch.median(distances))
        threshold = torch.quantile(distances, 1-threshold_quantile)
        mask = distances < threshold
        row_indices, col_indices = torch.where(mask)

        # Convert to global indices
        global_row_indices = torch.tensor(bucket_indices, device=vectors.device)[row_indices]
        global_col_indices = torch.tensor(bucket_indices, device=vectors.device)[col_indices]

        # Append results
        indices.append(torch.stack([global_row_indices, global_col_indices], dim=0))
        values.append(distances[mask])

    # Combine results into a sparse matrix
    if indices:
        indices = torch.cat(indices, dim=1)
        values = torch.cat(values)
    else:
        indices = torch.empty((2, 0), dtype=torch.int64, device=vectors.device)
        values = torch.empty(0, device=vectors.device)

    values = values.to(device='cpu')
    indices = indices.to(device='cpu')
    sparse_matrix = torch.sparse_coo_tensor(
        indices=indices,
        values=values,
        size=(vectors.size(0), vectors.size(0)),
    )
    return sparse_matrix

# Example usage
n_vectors = 1000
dim = 100
vectors = torch.randn(n_vectors, dim, device='mps')  # Random vectors on MPS

# Step 1: Partition vectors using LSH
num_hashes = 10
hash_size = 84
buckets, hash_keys = lsh_partition(vectors, num_hashes=num_hashes, hash_size=hash_size)

# Step 2: Compute sparse distance matrix within buckets
threshold_quantile = 0.85
sparse_matrix = compute_sparse_distances_from_buckets(vectors, buckets, threshold_quantile=threshold_quantile)

# Inspect sparse matrix
print(sparse_matrix)

tensor(indices=tensor([[ 12,  23, 317, 435, 468, 881, 121, 190, 193, 235, 458,
                        918, 934],
                       [ 12,  23, 317, 435, 468, 881, 121, 190, 193, 235, 458,
                        918, 934]]),
       values=tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
       size=(1000, 1000), nnz=13, layout=torch.sparse_coo)


In [65]:
# for bucket_indices in buckets.values():
#     if len(bucket_indices) > 1:
#         print(bucket_indices)

In [84]:
import torch
from transformers import GPT2LMHeadModel
import networkx as nx
import numpy as np
from scipy.spatial.distance import cosine

# Load GPT2 model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Get unembedding matrix (word embeddings)
unembedding_matrix = model.lm_head.weight.detach()

In [85]:
from transformers import GPT2Tokenizer

# Initialize the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Filter token positions to exclude non-latin, non-numeric, and special characters
def filter_token_positions():
    # Get the vocabulary from the tokenizer
    vocab = tokenizer.get_vocab()
    
    # Define a function to check if a token is valid
    def is_valid_token(token):
        return all(c.isalpha() and c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ" or c in "!@#$%^&*()[]{}/\\|<>;:,. " for c in token)

    # Filter the token positions
    valid_token_indices = torch.tensor([i for token,i in vocab.items() if is_valid_token(token)])
    
    return valid_token_indices

# Get filtered token positions
filtered_token_positions = filter_token_positions()

In [86]:
vocab = tokenizer.get_vocab()
filtered_vocab = {token: i for token, i in vocab.items() if i in filtered_token_positions}
# sort by length 
filtered_vocab = sorted(filtered_vocab.items(), key=lambda x: len(x[0]), reverse=True)

In [87]:
unembedding_matrix = unembedding_matrix[filtered_token_positions]

num_hashes = 7
hash_size = 84
buckets, hash_keys = lsh_partition(unembedding_matrix, num_hashes=num_hashes, hash_size=hash_size)

# Step 2: Compute sparse distance matrix within buckets
threshold_quantile = 0.85
sparse_similarity_matrix = compute_sparse_distances_from_buckets(unembedding_matrix, buckets, threshold_quantile=threshold_quantile)

print('percentage of non-zero connections:', sparse_similarity_matrix.coalesce().indices().shape[1]/unembedding_matrix.shape[0]**2)

percentage of non-zero connections: 0.0026365131103747727


In [88]:
sparse_similarity_matrix

tensor(indices=tensor([[    0,     0,     0,  ...,  9897, 11542, 13516],
                       [    0,    11,    41,  ...,  9897, 11542, 13516]]),
       values=tensor([0.0000, 2.9365, 3.8032,  ..., 0.0000, 0.0000, 0.0000]),
       size=(15082, 15082), nnz=599719, layout=torch.sparse_coo)

In [89]:
sparse_similarity_matrix.coalesce().indices().to('cpu').numpy().T[0]

array([0, 0])

In [90]:
# create graph
G = nx.Graph()
G.add_nodes_from(range(unembedding_matrix.shape[0]))
indices = sparse_similarity_matrix.coalesce().indices().to('cpu').numpy().T
values = sparse_similarity_matrix.coalesce().values().to('cpu').numpy().T
for i,j,w in zip(indices[:,0], indices[:,1], values):
    G.add_edge(i,j, weight=w)

In [91]:
from geomechinterp.curvature.balanced_forman_curvature import balanced_forman_curvature

# get adjacency matrix
A = nx.adjacency_matrix(G)
A = A.todense()
A = (A > 0).astype(int)

In [108]:
for pattern, bucket_indices in buckets.items():
    if len(bucket_indices) > 100 and len(bucket_indices) < 200:
        print(pattern)


(1, -1, -1, 1, 1, -1, -1)
(1, 1, -1, 1, -1, -1, -1)
(1, 1, -1, 1, 1, -1, -1)
(1, -1, -1, 1, -1, -1, -1)
(-1, -1, -1, 1, 1, 1, 1)
(1, 1, -1, 1, 1, 1, -1)
(-1, -1, 1, 1, 1, 1, 1)
(-1, 1, -1, 1, 1, 1, 1)
(-1, 1, 1, 1, 1, 1, 1)


In [93]:
# import seaborn as sns
# from matplotlib import pyplot as plt
# plt.imshow(A)

In [94]:
hash_keys.shape

(15082, 7)

In [114]:
# consider a single bucket
pattern = (1, -1, -1, 1, 1, -1, -1)
bucket_indices = buckets[pattern]
As = A[bucket_indices][:,bucket_indices]

vocab_subset = [filtered_vocab[i][0] for i in bucket_indices]

In [115]:
As.shape, len(vocab_subset)

((133, 133), 133)

In [ ]:
vocab_subset

In [110]:
C = balanced_forman_curvature(As)

In [ ]:
sns.clustermap(C)

In [117]:
# make pyvis graph with edges colored by curvature
from pyvis.network import Network
import numpy as np

# Create network
net = Network(notebook=True, height="750px", width="100%")

# set the physics layout of the network
net.barnes_hut()
net.show_buttons(filter_=['physics'])

# Add nodes
for i in range(len(As)):
    net.add_node(i, label=vocab_subset[i], title=vocab_subset[i])

# Add edges with curvature-based colors
for i in range(len(C)):
    for j in range(i+1, len(C)):
        if As[i,j] == 1:
            # Map curvature to color (red for negative, blue for positive)
            curvature = C[i,j]
            if curvature < 0:
                color = f'rgb(255,{int(255*(1+curvature))},{int(255*(1+curvature))})'
            else:
                color = f'rgb({int(255*(1-curvature))},{int(255*(1-curvature))},255)'
            
            net.add_edge(i, j, color=color, title=f'Curvature: {curvature:.3f}')

# Display the network
net.show('curvature_network.html')


curvature_network.html


In [118]:
### Dimensionality reduction
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=15, learning_rate=200, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(unembedding_matrix)



In [126]:
# make knn graph from umap projections
from sklearn.neighbors import kneighbors_graph
from umap import UMAP

umap = UMAP(n_components=20, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = umap.fit_transform(unembedding_matrix)

knn_graph = kneighbors_graph(X_umap, n_neighbors=100, mode='connectivity')
knn_graph = knn_graph.toarray()
knn_graph = (knn_graph > 0).astype(int)

/Users/solar/miniconda3/envs/pytorch/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [127]:
knn_graph

array([100, 100, 100, ..., 100, 100, 100])